In [ ]:
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"
#r "nuget: MathNet.Numerics, 5.0.0"

using System;
using System.Collections.Generic;
using System.Net.Http;
using System.IO;
using System.Linq;
using System.Globalization;
using Plotly.NET;
using Plotly.NET.LayoutObjects;
using Plotly.NET.Interactive;
using Plotly.NET.CSharp;
using Chart = Plotly.NET.CSharp.Chart;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.LinearAlgebra.Double;

string URL = "https://raw.githubusercontent.com/dbdmg/data-science-lab/master/datasets/mnist_test.csv";
HttpClient client = new HttpClient();
Stream stream = await client.GetStreamAsync(URL);
StreamReader reader = new StreamReader(stream);
string res = await reader.ReadToEndAsync();

var lines = res.Split('\n', StringSplitOptions.RemoveEmptyEntries).Select(l => l.Trim()).ToArray();
int rows = lines.Length;
int cols = 784;
double[,] X = new double[rows, cols];
int[] y = new int[rows];

for(int i = 0; i < rows; i++) {
    var parts = lines[i].Split(',');
    y[i] = int.Parse(parts[0]);
    for(int j = 0; j < cols; j++)
        X[i,j] = double.Parse(parts[j+1], CultureInfo.InvariantCulture) / 255.0;
}

Console.WriteLine($"Size matrix X: {rows} x {cols}");

Random random = new Random();
var previewCharts = new List<GenericChart>();
for (int k = 0; k < 10; k++) {
    int idx = random.Next(rows);
    double[][] matrix = new double[28][];
    for (int i = 0; i < 28; i++) {
        matrix[i] = new double[28];
        for (int j = 0; j < 28; j++) matrix[i][j] = X[idx, i * 28 + j];
    }
    previewCharts.Add(Chart.Heatmap<double,double,double,double>(zData: matrix).WithTitle($"Digit: {y[idx]}"));
}
display(Chart.Grid(previewCharts, 2, 5));

var matrixX = DenseMatrix.OfArray(X);
var columnMeans = matrixX.ColumnSums() / matrixX.RowCount;
var X_centered = DenseMatrix.OfRowVectors(matrixX.EnumerateRows().Select(row => row - columnMeans).ToArray());

var svd = X_centered.Svd(true);
var V = svd.VT.Transpose(); 
var S = svd.S;

var eigenvalues = S.PointwisePower(2);
var totalVar = eigenvalues.Sum();
var varRatio = eigenvalues.Take(3).Select(v => (v / totalVar) * 100).ToArray();

Console.WriteLine($"PC1: {varRatio[0]:F2}%, PC2: {varRatio[1]:F2}%, PC3: {varRatio[2]:F2}%");

var V_3 = V.SubMatrix(0, V.RowCount, 0, 3);
var X_pca = X_centered * V_3;

string[] palette = new string[] { "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf" };
var pointColors = y.Select(val => Color.fromHex(palette[val % 10])).ToArray();
var markerColor = Color.fromColors(pointColors);

var pcaChart = Chart.Scatter3D<double, double, double, string>(
    x: X_pca.Column(0).ToArray(),
    y: X_pca.Column(1).ToArray(),
    z: X_pca.Column(2).ToArray(),
    mode: StyleParam.Mode.Markers
).WithMarkerStyle(Size: 2, Opacity: 0.6, Color: markerColor).WithTitle("MNIST PCA 3D Clusters");

display(pcaChart);

var X_rec_matrix = X_pca * V_3.Transpose();

var origCharts = new List<GenericChart>();
var reconCharts = new List<GenericChart>();

for (int k = 0; k < 10; k++) {
    int rIdx = random.Next(rows);
    double[][] origImg = new double[28][];
    double[][] reconImg = new double[28][];
    for(int i=0; i<28; i++) {
        origImg[i] = matrixX.Row(rIdx).Skip(i*28).Take(28).ToArray();
        reconImg[i] = (X_rec_matrix.Row(rIdx) + columnMeans).Skip(i*28).Take(28).ToArray();
    }
    origCharts.Add(Chart.Heatmap<double,double,double,double>(origImg).WithTitle($"Orig: {y[rIdx]}"));
    reconCharts.Add(Chart.Heatmap<double,double,double,double>(reconImg).WithTitle($"Rec: {y[rIdx]}"));
}

var allReconCharts = new List<GenericChart>();
allReconCharts.AddRange(origCharts);
allReconCharts.AddRange(reconCharts);

display(Chart.Grid(allReconCharts, 2, 10));

int[] kValues = new int[] { 1, 5, 10, 20, 50, 100, 200, 784 };
var mseList = new List<double>();
var expVarList = new List<double>();

foreach (int kNum in kValues) {
    double currentVar = eigenvalues.Take(kNum).Sum() / totalVar * 100;
    expVarList.Add(currentVar);

    var V_k = V.SubMatrix(0, V.RowCount, 0, kNum);
    var X_pca_k = X_centered * V_k;
    var X_rec_k = X_pca_k * V_k.Transpose();

    double totalSqErrorK = 0;
    for (int i = 0; i < rows; i++) {
        totalSqErrorK += (X_centered.Row(i) - X_rec_k.Row(i)).PointwisePower(2).Sum();
    }
    mseList.Add(totalSqErrorK / (rows * cols));
}

var mseGraph = Chart.Line<int, double, string>(x: kValues, y: mseList)
    .WithTitle("MSE vs k")
    .WithXAxisStyle(Title.init("Number of Components"))
    .WithYAxisStyle(Title.init("Mean Squared Error"));

var varGraph = Chart.Line<int, double, string>(x: kValues, y: expVarList)
    .WithTitle("Explained Variance (%) vs k")
    .WithXAxisStyle(Title.init("Number of Components"))
    .WithYAxisStyle(Title.init("Explained Variance (%)"));

display(Chart.Grid(new List<GenericChart> { mseGraph, varGraph }, 1, 2));

Installed Packages MathNet.Numerics, 5.0.0 Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Розмір матриці X: 10000 x 784
Унікальні класи: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9


<!-- Plotly chart will be drawn inside this DIV -->

PC1: 10.05%, PC2: 7.54%, PC3: 6.14%


<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->